# Visium Spatial Gallery

Spatial plots colored by Leiden clustering for all models (topomics variants + baselines).

In [ ]:
from pathlib import Path
import numpy as np
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt


def compute_moran_i(values, spatial_weights):
    """Compute Moran's I statistic for a 1D array of values."""
    n = len(values)
    z = values - values.mean()
    W = spatial_weights.tocsr()
    numerator = z @ W @ z
    denominator = z @ z
    if denominator == 0:
        return 0.0
    S0 = W.sum()
    if S0 == 0:
        return 0.0
    return (n / S0) * (numerator / denominator)


def plot_spatial_leiden(theta, spatial_coords, spatial_weights, adata, title):
    """Compute Leiden from latent, plot spatial scatter colored by cluster."""
    morans_values = [compute_moran_i(theta[:, i], spatial_weights) for i in range(theta.shape[1])]
    mean_morans = np.mean(morans_values)

    adata.obsm["X_topic"] = theta - 1
    sc.pp.neighbors(adata, metric="cosine", use_rep="X_topic", key_added="topic_neighbors")
    sc.tl.leiden(adata, neighbors_key="topic_neighbors", key_added="leiden")

    leiden_encoded = adata.obs["leiden"].astype("category").cat.codes.to_numpy()

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(
        spatial_coords[:, 0],
        spatial_coords[:, 1],
        c=leiden_encoded,
        cmap="tab20",
        s=30,
        alpha=0.8,
        linewidths=0,
    )
    ax.set_title(f"{title} | Moran's I = {mean_morans:.4f}")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_aspect("equal")
    ax.invert_yaxis()
    plt.show()


# ---- Load Visium dataset (same as training) ----
adata = sq.datasets.visium_hne_adata()
adata.X = adata.raw.X
if "spatial_connectivities" not in adata.obsp:
    sq.gr.spatial_neighbors(adata, coord_type="generic")

spatial_coords = adata.obsm["spatial"]
spatial_weights = adata.obsp["spatial_connectivities"]

# ---- Omics-Topic models ----
base_dirs = [
    Path("/data/omics_topic_models/sctm_comparison"),
    Path("/data/omics_topic_models/sctm_comparison_graphconv"),
    Path("/data/omics_topic_models/sctm_comparison_meanfield"),
]

run_dirs = []
for base in base_dirs:
    if not base.exists():
        print(f"Warning: {base} does not exist")
        continue
    for d in sorted(base.iterdir()):
        if d.is_dir() and d.name.startswith("prior_"):
            latent_path = d / "latent_representation.npy"
            if latent_path.exists():
                run_dirs.append(d)

print(f"Found {len(run_dirs)} topomics model runs")

for run_dir in run_dirs:
    display_name = f"{run_dir.parent.name}/{run_dir.name}"
    print(f"=== {display_name} ===")
    theta = np.load(run_dir / "latent_representation.npy")
    plot_spatial_leiden(theta, spatial_coords, spatial_weights, adata, display_name)

# ---- Baseline models ----
BASELINES_DIR = Path("/data/omics_topic_models/sctm_comparison/baselines")
baseline_files = {
    "scVI": "latent_scvi.npy",
    "STAMP": "latent_stamp.npy",
    "AmortizedLDA": "latent_amortized_lda.npy",
}

print(f"\n{'='*60}")
print("Baseline models")
print(f"{'='*60}")

for name, fname in baseline_files.items():
    latent_path = BASELINES_DIR / fname
    if not latent_path.exists():
        print(f"  {name}: not found ({latent_path})")
        continue
    print(f"=== {name} ===")
    theta = np.load(latent_path)
    plot_spatial_leiden(theta, spatial_coords, spatial_weights, adata, name)